# Ropedia Academy — A5 · Hand & face models (MANO, FLAME)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A5.ipynb)

> **Shows MANO/FLAME are SMPL specialized via one shared parametric_model, composes them into SMPL-X, and computes a mesh-level contact distance (why 3D > a box).**
>
> 展示 MANO/FLAME 是用同一个 parametric_model 特化的 SMPL，组合成 SMPL-X，并计算网格级接触距离（为何 3D 胜过框）。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A5

In [ ]:
import torch

# ONE recipe, specialized: identity blendshapes + articulation -> body / hand / face.
def parametric_model(beta, pose, template, shape_dirs, articulate):
    v = template + shape_dirs @ beta            # identity / shape
    return articulate(v, pose)                   # pose it (LBS / jaw / expression)

def articulate(v, pose):                          # toy: rotate verts about z by pose[0]
    a = pose[0]; c, s = torch.cos(a), torch.sin(a)
    R = torch.tensor([[c,-s,0],[s,c,0],[0,0,1.]])
    return v @ R.T

mk = lambda Vn, nb: (torch.randn(Vn,3), torch.randn(Vn,3,nb))
hand = parametric_model(torch.randn(10), torch.randn(3), *mk(40,10), articulate)  # MANO
face = parametric_model(torch.randn(50), torch.randn(3), *mk(40,50), articulate)  # FLAME
body = torch.randn(60, 3)
smplx = torch.cat([body, hand, face], 0)         # SMPL-X = body + hands + face
print("SMPL-X mesh vertices:", smplx.shape[0])

# Why a 3D hand MESH (not a 2D box): contact = surface proximity to the object.
obj = torch.tensor([0.2, 0.1, 0.0])
print("hand-object contact distance:", round((hand - obj).norm(dim=1).min().item(), 3))

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A5
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks